# 04 · Train the adapter

**Examples → prediction → loss → adapter update**

Use the practice data from notebook 02 and the CPU adapter setup from notebook 03. We train for **40 updates**. The large Lightning models keep serving as before.

**Who does what:** Nano coordinates the tools. PyTorch trains the tiny CPU adapter. Lightning is not called or trained in this lesson.

###  optimization and validation loss

PyTorch's [optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) connects loss, backpropagation, and optimizer updates. The [PEFT guide](https://huggingface.co/docs/peft/main/en/quicktour) explains adapter parameters. Match the [tested versions](README.md#concept-sdks-and-documentation).

finish manual 02 and 03, then load → batch → baseline → train. This runs 40 real CPU updates and saves `cpu/student-adapter` and measured metrics. No Lightning generation is needed in the manual exercise.

**Open the actual implementation:** [04 lesson source](agent/lessons/04-train-adapter.json), especially the `batch`, `evaluate`, and optimizer-loop definitions. `lesson.step` executes fixed source cells; [the execution guide](README.md#how-the-notebook-helpers-execute) explains where the code runs.

**Read the result carefully:** the Transformers model computes the causal-language-model loss from masked labels. Context/padding are ignored; gradients update the adapter. Lower loss is a diagnostic, not proof of useful generated answers. Notebook 06 tests those answers.

The agent uses a workshop-authored training skill. See [Skills and training](README.md#skills-and-training) for the distinction between these instructions, NeMo libraries, and the separate NeMo Skills pipeline.


### Start here
Run the short cells in order. Helpers execute the existing lesson code and keep detailed logs expandable. All displayed results come from your run. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import LessonWalkthrough, table, bars, loss_chart, show_skill_library
lesson = LessonWalkthrough('04-train-adapter')

### 1 · Load the starting point

The saved base and adapter recipe become a fresh training model. The setup prints the exact input folders; complete the manual sections of notebooks 02 and 03 first.

In [ ]:
lesson.step("load")
print("Training device:", lesson.ns["device"])

### 2 · See a training batch

Context and padding are ignored by the loss; answer tokens are scored. This practice model uses byte IDs, unlike Lightning’s tokenizer.

In [ ]:
lesson.step("batch")
batch = lesson.ns["example_batch"]
table(["Batch component", "Shape"], [(k, tuple(v.shape)) for k,v in batch.items()])
print("Answer tokens scored:", int((batch["labels"] != -100).sum()))

### 3 · Measure before training

Validation examples stay outside training. First measure how well the untrained adapter predicts their answers.

In [ ]:
lesson.step("baseline")
print("Validation loss before:", lesson.ns["before"])

### API pattern · the optimizer step inside the helper

Adapted from PyTorch's [optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) to our Transformers model. Reference only: the next cell runs the actual 40-step recipe, which also accumulates gradients and checks their norm.

```python
optimizer.zero_grad()
loss = model(**example_batch).loss
loss.backward()
optimizer.step()
```

`model` computes next-token loss, including the causal shift. The optional AutoModel training script consumes already-shifted `ChatDataset` labels and computes its loss differently. Do not copy one loss calculation into the other without checking alignment.


### 4 · Train and watch the loss

Run 40 real CPU optimizer updates, then save the adapter. The curve is measured training loss; the bars compare validation loss before and after.

Lower loss does not prove good generated answers. The existing lesson checks a sample from one frozen tensor; it is not a full-weight audit.

In [ ]:
lesson.step("train")
metrics = lesson.read("cpu/training-metrics.json")
loss_chart(metrics["history"])
bars({"Validation before": metrics["validation_before"], "Validation after": metrics["validation_after"]})

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
manual_result = lesson.finish()
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-train-adapter**. Open the actual instructions below.

**`inspect-inputs` → `train-adapter` → `check-training`**

In [ ]:
show_skill_library('04-train-adapter')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-train-adapter/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-train-adapter/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `02-training-data`, `03-build-lora`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('04-train-adapter')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-train-adapter/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-inputs', 'train-adapter', 'check-training']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the tools. PyTorch trains the tiny CPU adapter. Lightning is not called or trained in this lesson.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** 40 training steps completed; adapter saved; frozen-weight check passed.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** The adapter has learned from examples. Next, reload it and inspect what actually changed.

Exit the TUI with `/exit` when done.